In [1]:
import gymnasium as gym
from gymnasium import spaces
import numpy as np
import simpy
from LEOEnvironmentRL import initialize, load_route_from_csv  # Use RL version
import pandas as pd
import os
from stable_baselines3 import DQN
from sb3_contrib import MaskablePPO
from sb3_contrib.common.wrappers import ActionMasker
import torch
import random
import matplotlib.pyplot as plt
import plotly.graph_objects as go
# %% 
import sb3_contrib
from HandoverEnvironment import LEOEnv as LEOEnvPPO 
from HandoverEnvironment import mask_fn, predict_valid_action
from HandoverEnvironment_DQN import LEOEnv as LEOEnvDQN
from HandoverEnvironment_DQN import predict_valid_action as predict_valid_action_dqn
from ODT import LEOEnv as LEOEnvODT
from ODT import predict_valid_action_dt
from ODT import OnlineDecisionTransformer
from LEOEnvironment import LEOEnv as LEOEnvBase

['ARS', 'CrossQ', 'MaskablePPO', 'QRDQN', 'RecurrentPPO', 'TQC', 'TRPO', '__all__', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__', '__version__', 'ars', 'common', 'crossq', 'file_handler', 'os', 'ppo_mask', 'ppo_recurrent', 'qrdqn', 'tqc', 'trpo', 'version_file']


In [2]:
import os

scenarios = ["no_scenario", "load_cycle_1", "load_cycle_2", "load_cycle_5", "medium_aircraft", "large_aircraft", "snr_congested"]
handover_idx = 6

for scenario in scenarios:
    base_path = f"BASELINE_observations_{scenario}.csv"
    ppo_path = f"PPO_observations_{scenario}.csv"
    dqn_path = f"DQN_observations_{scenario}.csv"
    odt_path = f"ODT_observations_{scenario}.csv"
    odt_finetuned_path = f"ODT_FINETUNED_observations_{scenario}.csv"

    if not (os.path.exists(base_path) and os.path.exists(ppo_path) and os.path.exists(dqn_path) and os.path.exists(odt_path) and os.path.exists(odt_finetuned_path)):
        print(f"Missing files for scenario '{scenario}', skipping.")
        continue

    obs_base = pd.read_csv(base_path).values
    obs_ppo = pd.read_csv(ppo_path).values
    obs_dqn = pd.read_csv(dqn_path).values
    obs_odt = pd.read_csv(odt_path).values
    obs_odt_finetuned = pd.read_csv(odt_finetuned_path).values

    fig = go.Figure()
    fig.add_trace(go.Scatter(y=obs_base[:, handover_idx], mode='lines', name='Baseline'))
    fig.add_trace(go.Scatter(y=obs_ppo[:, handover_idx], mode='lines', name='PPO'))
    fig.add_trace(go.Scatter(y=obs_dqn[:, handover_idx], mode='lines', name='DQN'))
    fig.add_trace(go.Scatter(y=obs_odt[:, handover_idx], mode='lines', name='ODT'))
    fig.add_trace(go.Scatter(y=obs_odt_finetuned[:, handover_idx], mode='lines', name='ODT Finetuned'))
    #fig.add_trace(go.Scatter(y=obs_odt_finetuned[:, handover_idx], mode='lines', name='ODT Finetuned'))
    fig.update_layout(
        title=f"Total Handovers Over Time - {scenario}",
        xaxis_title='Step',
        yaxis_title='Total Handovers',
        legend_title='Agent',
        template='plotly_white'
    )
    fig.show()


In [3]:
import os

scenarios = ["no_scenario", "load_cycle_1", "load_cycle_2", "load_cycle_5", "medium_aircraft", "large_aircraft", "snr_congested"]
throughput_idx = 13
requested_throughput_idx = 10

for scenario in scenarios:
    base_path = f"BASELINE_observations_{scenario}.csv"
    ppo_path = f"PPO_observations_{scenario}.csv"
    dqn_path = f"DQN_observations_{scenario}.csv"
    odt_path = f"ODT_observations_{scenario}.csv"
    odt_finetuned_path = f"ODT_FINETUNED_observations_{scenario}.csv"

    if not (os.path.exists(base_path) and os.path.exists(ppo_path) and os.path.exists(dqn_path) and os.path.exists(odt_path) and os.path.exists(odt_finetuned_path)):
        print(f"Missing files for scenario '{scenario}', skipping.")
        continue

    obs_base = pd.read_csv(base_path).values
    obs_ppo = pd.read_csv(ppo_path).values
    obs_dqn = pd.read_csv(dqn_path).values
    obs_odt = pd.read_csv(odt_path).values
    obs_odt_finetuned = pd.read_csv(odt_finetuned_path).values

    fig = go.Figure()
    fig.add_trace(go.Scatter(y=obs_ppo[:, requested_throughput_idx], mode='lines', name='Requested'))
    fig.add_trace(go.Scatter(y=obs_base[:, throughput_idx], mode='lines', name='Baseline'))
    fig.add_trace(go.Scatter(y=obs_ppo[:, throughput_idx], mode='lines', name='PPO'))
    fig.add_trace(go.Scatter(y=obs_dqn[:, throughput_idx], mode='lines', name='DQN'))
    fig.add_trace(go.Scatter(y=obs_odt[:, throughput_idx], mode='lines', name='ODT'))
    fig.add_trace(go.Scatter(y=obs_odt_finetuned[:, throughput_idx], mode='lines', name='ODT Finetuned'))
    #fig.add_trace(go.Scatter(y=obs_odt_finetuned[:, throughput_idx], mode='lines', name='ODT Finetuned'))
    fig.update_layout(
        title=f"Average Throughput Over Time - {scenario}",
        xaxis_title='Step',
        yaxis_title='Average Throughput',
        legend_title='Agent',
        template='plotly_white'
    )
    fig.show()


In [4]:
import os

scenarios = ["no_scenario", "load_cycle_1", "load_cycle_2", "load_cycle_5", "medium_aircraft", "large_aircraft", "snr_congested"]
allocated_bw_idx = 7
demand_bw_idx = 9 

for scenario in scenarios:
    base_path = f"BASELINE_observations_{scenario}.csv"
    ppo_path = f"PPO_observations_{scenario}.csv"
    dqn_path = f"DQN_observations_{scenario}.csv"
    odt_path = f"ODT_observations_{scenario}.csv"
    odt_finetuned_path = f"ODT_FINETUNED_observations_{scenario}.csv"

    if not (os.path.exists(base_path) and os.path.exists(ppo_path) and os.path.exists(dqn_path) and os.path.exists(odt_path) and os.path.exists(odt_finetuned_path)):
        print(f"Missing files for scenario '{scenario}', skipping.")
        continue

    obs_base = pd.read_csv(base_path).values
    obs_ppo = pd.read_csv(ppo_path).values
    obs_dqn = pd.read_csv(dqn_path).values
    obs_odt = pd.read_csv(odt_path).values
    obs_odt_finetuned = pd.read_csv(odt_finetuned_path).values

    fig = go.Figure()
    fig.add_trace(go.Scatter(y=obs_base[:, demand_bw_idx], mode='lines', name='Demand'))
    fig.add_trace(go.Scatter(y=obs_base[:, allocated_bw_idx], mode='lines', name='Baseline'))
    fig.add_trace(go.Scatter(y=obs_ppo[:, allocated_bw_idx], mode='lines', name='PPO'))
    fig.add_trace(go.Scatter(y=obs_dqn[:, allocated_bw_idx], mode='lines', name='DQN'))
    fig.add_trace(go.Scatter(y=obs_odt[:, allocated_bw_idx], mode='lines', name='ODT'))
    fig.add_trace(go.Scatter(y=obs_odt_finetuned[:, allocated_bw_idx], mode='lines', name='ODT Finetuned'))
    #fig.add_trace(go.Scatter(y=obs_odt_finetuned[:, allocated_bw_idx], mode='lines', name='ODT Finetuned'))
    fig.update_layout(
        title=f"Allocated Bandwidth Over Time - {scenario}",
        xaxis_title='Step',
        yaxis_title='Allocated Bandwidth',
        legend_title='Agent',
        template='plotly_white'
    )
    fig.show()


In [5]:
import os

scenarios = ["no_scenario", "load_cycle_1", "load_cycle_2", "load_cycle_5", "medium_aircraft", "large_aircraft", "snr_congested"]
allocated_demand_idx = 8

for scenario in scenarios:
    base_path = f"BASELINE_observations_{scenario}.csv"
    ppo_path = f"PPO_observations_{scenario}.csv"
    dqn_path = f"DQN_observations_{scenario}.csv"
    odt_path = f"ODT_observations_{scenario}.csv"
    odt_finetuned_path = f"ODT_FINETUNED_observations_{scenario}.csv"

    if not (os.path.exists(base_path) and os.path.exists(ppo_path) and os.path.exists(dqn_path) and os.path.exists(odt_path) and os.path.exists(odt_finetuned_path)):
        print(f"Missing files for scenario '{scenario}', skipping.")
        continue

    obs_base = pd.read_csv(base_path).values
    obs_ppo = pd.read_csv(ppo_path).values
    obs_dqn = pd.read_csv(dqn_path).values
    obs_odt = pd.read_csv(odt_path).values
    obs_odt_finetuned = pd.read_csv(odt_finetuned_path).values

    avg_allocated_to_demand_base = []
    avg_allocated_to_demand_ppo = []
    avg_allocated_to_demand_dqn = []
    avg_allocated_to_demand_odt = []
    avg_allocated_to_demand_odt_finetuned = []

    for i in range(len(obs_base)):
        avg_allocated_to_demand_base.append(np.mean(obs_base[:i, allocated_demand_idx]))
        avg_allocated_to_demand_ppo.append(np.mean(obs_ppo[:i, allocated_demand_idx]))
        avg_allocated_to_demand_dqn.append(np.mean(obs_dqn[:i, allocated_demand_idx]))
        avg_allocated_to_demand_odt.append(np.mean(obs_odt[:i, allocated_demand_idx]))
        #avg_allocated_to_demand_odt_finetuned.append(np.mean(obs_odt_finetuned[:i, allocated_demand_idx]))

    fig = go.Figure()
    fig.add_trace(go.Scatter(y=avg_allocated_to_demand_base, mode='lines', name='Baseline'))
    fig.add_trace(go.Scatter(y=avg_allocated_to_demand_ppo, mode='lines', name='PPO'))
    fig.add_trace(go.Scatter(y=avg_allocated_to_demand_dqn, mode='lines', name='DQN'))
    fig.add_trace(go.Scatter(y=avg_allocated_to_demand_odt, mode='lines', name='ODT'))
    fig.add_trace(go.Scatter(y=avg_allocated_to_demand_odt_finetuned, mode='lines', name='ODT Finetuned'))
    #fig.add_trace(go.Scatter(y=avg_allocated_to_demand_odt_finetuned, mode='lines', name='ODT Finetuned'))
    fig.update_layout(
        title=f"Allocation to Demand Over Time - {scenario}",
        xaxis_title='Step',
        yaxis_title='Allocated to Demand',
        legend_title='Agent',
        template='plotly_white'
    )
    fig.show()


/Users/hindmukhtar/Library/Python/3.9/lib/python/site-packages/numpy/core/fromnumeric.py:3464: RuntimeWarning:

Mean of empty slice.

/Users/hindmukhtar/Library/Python/3.9/lib/python/site-packages/numpy/core/_methods.py:192: RuntimeWarning:

invalid value encountered in scalar divide



/Users/hindmukhtar/Library/Python/3.9/lib/python/site-packages/numpy/core/fromnumeric.py:3464: RuntimeWarning:

Mean of empty slice.

/Users/hindmukhtar/Library/Python/3.9/lib/python/site-packages/numpy/core/_methods.py:192: RuntimeWarning:

invalid value encountered in scalar divide



/Users/hindmukhtar/Library/Python/3.9/lib/python/site-packages/numpy/core/fromnumeric.py:3464: RuntimeWarning:

Mean of empty slice.

/Users/hindmukhtar/Library/Python/3.9/lib/python/site-packages/numpy/core/_methods.py:192: RuntimeWarning:

invalid value encountered in scalar divide



/Users/hindmukhtar/Library/Python/3.9/lib/python/site-packages/numpy/core/fromnumeric.py:3464: RuntimeWarning:

Mean of empty slice.

/Users/hindmukhtar/Library/Python/3.9/lib/python/site-packages/numpy/core/_methods.py:192: RuntimeWarning:

invalid value encountered in scalar divide



/Users/hindmukhtar/Library/Python/3.9/lib/python/site-packages/numpy/core/fromnumeric.py:3464: RuntimeWarning:

Mean of empty slice.

/Users/hindmukhtar/Library/Python/3.9/lib/python/site-packages/numpy/core/_methods.py:192: RuntimeWarning:

invalid value encountered in scalar divide



/Users/hindmukhtar/Library/Python/3.9/lib/python/site-packages/numpy/core/fromnumeric.py:3464: RuntimeWarning:

Mean of empty slice.

/Users/hindmukhtar/Library/Python/3.9/lib/python/site-packages/numpy/core/_methods.py:192: RuntimeWarning:

invalid value encountered in scalar divide



/Users/hindmukhtar/Library/Python/3.9/lib/python/site-packages/numpy/core/fromnumeric.py:3464: RuntimeWarning:

Mean of empty slice.

/Users/hindmukhtar/Library/Python/3.9/lib/python/site-packages/numpy/core/_methods.py:192: RuntimeWarning:

invalid value encountered in scalar divide



In [6]:
# Average Allocation to demand 
print("Baseline Average Allocation to Demand: ", np.mean(obs_base[:, 8]))
print("PPO Average Allocation to Demand: ", np.mean(obs_ppo[:, 8]))
print("DQN Average Allocation to Demand: ", np.mean(obs_dqn[:, 8]))
print("ODT Average Allocation to Demand: ", np.mean(obs_odt[:, 8]))
#print("ODT Finetuned Average Allocation to Demand: ", np.mean(obs_odt_finetuned[:, 8]))    


Baseline Average Allocation to Demand:  0.876170172736517
PPO Average Allocation to Demand:  0.9627928028706041
DQN Average Allocation to Demand:  0.9785044479722677
ODT Average Allocation to Demand:  0.9655654461238389


In [7]:
# Average throughput across runs
print("Baseline Average Throughput Mbps: ", np.mean(obs_base[:, 13]))
print("PPO Average Throughput Mbps: ", np.mean(obs_ppo[:, 13]))
print("DQN Average Throughput Mbps: ", np.mean(obs_dqn[:, 13]))
print("ODT Average Throughput Mbps: ", np.mean(obs_odt[:, 13]))
#print("ODT Finetuned Average Throughput Mbps: ", np.mean(obs_odt_finetuned[:, 13]))

Baseline Average Throughput Mbps:  17.521939356093487
PPO Average Throughput Mbps:  18.693297163836398
DQN Average Throughput Mbps:  18.958823966230792
ODT Average Throughput Mbps:  18.71483136038683


In [8]:
# Average Delay across runs
print("Baseline Average Delay ms: ", np.mean(obs_base[:, 14]))
print("PPO Average Delay ms: ", np.mean(obs_ppo[:, 14]))
print("DQN Average Delay ms: ", np.mean(obs_dqn[:, 14]))
print("ODT Average Delay ms: ", np.mean(obs_odt[:, 14]))
#print("ODT Finetuned Average Delay ms: ", np.mean(obs_odt_finetuned[:, 14]))

Baseline Average Delay ms:  0.17384473822473814
PPO Average Delay ms:  0.17384473822473814
DQN Average Delay ms:  0.17384473822473814
ODT Average Delay ms:  0.17384473822473814


In [9]:
# Total Handovers 
print("Baseline Total Handovers: ", np.max(obs_base[:, 6]))
print("PPO Total Handovers: ", np.max(obs_ppo[:, 6]))
print("DQN Total Handovers: ", np.max(obs_dqn[:, 6]))
print("ODT Total Handovers: ", np.max(obs_odt[:, 6]))
#print("ODT Finetuned Total Handovers: ", np.max(obs_odt_finetuned[:, 6]))

Baseline Total Handovers:  269.0
PPO Total Handovers:  253.0
DQN Total Handovers:  244.0
ODT Total Handovers:  260.0


In [10]:
import pandas as pd

scenarios = ["no_scenario", "load_cycle_1", "load_cycle_2", "load_cycle_5", "medium_aircraft", "large_aircraft", "snr_congested"]
agents = ["BASELINE", "PPO", "DQN", "ODT", "ODT_FINETUNED"]

# Column indices based on testscript output
allocated_idx = 7
ratio_idx = 8
demand_idx = 9
delay_idx = 11
throughput_idx = 12
handover_idx = 6
service_drop_idx = 16

rows = []
for scenario in scenarios:
    for agent in agents:
        path = f"{agent}_observations_{scenario}.csv"
        if not os.path.exists(path):
            continue
        data = pd.read_csv(path).values
        rows.append({
            "scenario": scenario,
            "agent": agent,
            "avg_allocated_MB": data[:, allocated_idx].mean(),
            "avg_allocation_ratio": data[:, ratio_idx].mean(),
            "avg_demand_MB": data[:, demand_idx].mean(),
            "avg_delay_s": data[:, delay_idx].mean(),
            "avg_throughput_mbps": data[:, throughput_idx].mean(),
            "total_handovers": data[:, handover_idx].max(),
            "total_service_drop_s": data[:, service_drop_idx].sum(),
        })

summary_df = pd.DataFrame(rows)
summary_df.sort_values(["scenario", "agent"], inplace=True)
summary_df


,scenario,agent,avg_allocated_MB,avg_allocation_ratio,avg_demand_MB,avg_delay_s,avg_throughput_mbps,total_handovers,total_service_drop_s
25,large_aircraft,BASELINE,48.485279,0.977246,49.503843,1.507384e+01,0.054807,257.0,75.104881
27,large_aircraft,DQN,49.290288,0.995590,49.503843,3.293025e-01,0.054745,248.0,0.000000
28,large_aircraft,ODT,48.961358,0.988812,49.503843,4.973209e+00,0.054760,254.0,24.688061
29,large_aircraft,ODT_FINETUNED,48.559814,0.979570,49.503843,1.421535e+01,0.054800,253.0,74.010345
26,large_aircraft,PPO,49.473667,0.999397,49.503843,4.948269e-03,0.054807,231.0,0.000000
5,load_cycle_1,BASELINE,11.845063,0.983589,12.031083,1.482582e+01,0.054807,257.0,75.104881
7,load_cycle_1,DQN,12.014198,0.998405,12.031083,5.806598e-02,0.054767,250.0,0.000000
8,load_cycle_1,ODT,11.846052,0.984542,12.031083,1.392128e+01,0.054753,254.0,74.010345
9,load_cycle_1,ODT_FINETUNED,11.894539,0.989163,12.031083,9.300210e+00,0.054659,255.0,49.322285
6,load_cycle_1,PPO,12.031083,1.000000,12.031083,6.807396e-17,0.054784,230.0,0.000000


In [11]:
# Metrics as rows; columns = (agent, scenario)
scenarios = ["no_scenario", "load_cycle_1", "load_cycle_2", "load_cycle_5", "medium_aircraft", "large_aircraft", "snr_congested"]
agents = ["BASELINE", "PPO", "DQN", "ODT", "ODT_FINETUNED"]
allocated_idx = 7
ratio_idx = 8
demand_idx = 9
delay_idx = 11
throughput_idx = 12
handover_idx = 6
service_drop_idx = 16
rows = []
for scenario in scenarios:
    for agent in agents:
        path = f"{agent}_observations_{scenario}.csv"
        if not os.path.exists(path):
            continue
        data = pd.read_csv(path).values
        rows.append({
            'scenario': scenario,
            'agent': agent,
            'avg_allocated_MB': data[:, allocated_idx].mean(),
            'avg_allocation_ratio': data[:, ratio_idx].mean(),
            'avg_demand_MB': data[:, demand_idx].mean(),
            'avg_delay_s': data[:, delay_idx].mean(),
            'avg_throughput_mbps': data[:, throughput_idx].mean(),
            'total_handovers': data[:, handover_idx].max(),
            'total_service_drop_s': data[:, service_drop_idx].sum(),
        })
summary_df = pd.DataFrame(rows)
long_df = summary_df.melt(id_vars=['agent', 'scenario'], var_name='metric', value_name='value')
pivot = long_df.pivot(index='metric', columns=['agent', 'scenario'], values='value')
pivot


agent,BASELINE,PPO,DQN,ODT,ODT_FINETUNED,BASELINE,PPO,DQN,ODT,ODT_FINETUNED,...,BASELINE,PPO,DQN,ODT,ODT_FINETUNED,BASELINE,PPO,DQN,ODT,ODT_FINETUNED
scenario,no_scenario,no_scenario,no_scenario,no_scenario,no_scenario,load_cycle_1,load_cycle_1,load_cycle_1,load_cycle_1,load_cycle_1,...,large_aircraft,large_aircraft,large_aircraft,large_aircraft,large_aircraft,snr_congested,snr_congested,snr_congested,snr_congested,snr_congested
metric,,,,,,,,,,,,,,,,,,,,,
avg_allocated_MB,11.845063,1.203108e+01,1.203108e+01,11.836888,11.846052,11.845063,1.203108e+01,12.014198,11.846052,11.894539,...,48.485279,49.473667,49.290288,48.961358,48.559814,10.927472,11.662539,11.827906,11.676190,11.679351
avg_allocation_ratio,0.983589,1.000000e+00,1.000000e+00,0.983673,0.984542,0.983589,1.000000e+00,0.998405,0.984542,0.989163,...,0.977246,0.999397,0.995590,0.988812,0.979570,0.876170,0.962793,0.978504,0.965565,0.965565
avg_delay_s,14.825817,6.807396e-17,6.807396e-17,13.928221,13.921282,14.825817,6.807396e-17,0.058066,13.921282,9.300210,...,15.073840,0.004948,0.329302,4.973209,14.215353,121.113624,35.183012,19.471367,32.410369,32.410369
avg_demand_MB,12.031083,1.203108e+01,1.203108e+01,12.031083,12.031083,12.031083,1.203108e+01,12.031083,12.031083,12.031083,...,49.503843,49.503843,49.503843,49.503843,49.503843,12.031083,12.031083,12.031083,12.031083,12.031083
avg_throughput_mbps,0.054807,5.476976e-02,5.475591e-02,0.054724,0.054774,0.054807,5.478354e-02,0.054767,0.054753,0.054659,...,0.054807,0.054807,0.054745,0.054760,0.054800,0.054889,0.054670,0.054680,0.054720,0.054804
total_handovers,257.000000,2.300000e+02,2.460000e+02,254.000000,262.000000,257.000000,2.300000e+02,250.000000,254.000000,255.000000,...,257.000000,231.000000,248.000000,254.000000,253.000000,269.000000,253.000000,244.000000,260.000000,248.000000
total_service_drop_s,75.104881,0.000000e+00,0.000000e+00,74.010345,74.010345,75.104881,0.000000e+00,0.000000,74.010345,49.322285,...,75.104881,0.000000,0.000000,24.688061,74.010345,656.178806,191.640367,106.417842,176.543947,176.292624


In [12]:
import os
import pandas as pd
import numpy as np
import plotly.graph_objects as go

scenarios = ["no_scenario", "load_cycle_1", "load_cycle_2", "load_cycle_5", "medium_aircraft", "large_aircraft", "snr_congested"]
agents = ["BASELINE", "PPO", "DQN", "ODT", "ODT_FINETUNED"]
handover_idx = 6
service_drop_idx = 16

def service_drop_per_handover(obs):
    handovers = obs[:, handover_idx].astype(int)
    drop = obs[:, service_drop_idx]
    handover_steps = np.where(np.diff(handovers) > 0)[0] + 1
    drops = []
    for i, step in enumerate(handover_steps):
        end = handover_steps[i + 1] if i + 1 < len(handover_steps) else len(drop)
        drops.append(drop[step:end].sum())
    return drops

rows = []
for scenario in scenarios:
    for agent in agents:
        path = f"{agent}_observations_{scenario}.csv"
        if not os.path.exists(path):
            continue
        data = pd.read_csv(path).values
        for drop in service_drop_per_handover(data):
            rows.append({
                'scenario': scenario,
                'agent': agent,
                'drop_per_handover_s': drop,
            })

drop_df = pd.DataFrame(rows)
drop_df

for scenario in scenarios:
    subset = drop_df[drop_df['scenario'] == scenario]
    if subset.empty:
        continue
    fig = go.Figure()
    for agent in agents:
        vals = subset[subset['agent'] == agent]['drop_per_handover_s']
        if vals.empty:
            continue
        fig.add_trace(go.Box(y=vals, name=agent, boxmean=True))
    fig.update_layout(
        title=f"Service Drop per Handover - {scenario}",
        yaxis_title='Service Drop per Handover (s)',
        xaxis_title='Agent',
        template='plotly_white'
    )
    fig.show()


In [13]:
import os
import pandas as pd
import numpy as np
import plotly.graph_objects as go

scenarios = ["no_scenario", "load_cycle_1", "load_cycle_2", "load_cycle_5", "medium_aircraft", "large_aircraft", "snr_congested"]
agents = ["BASELINE", "PPO", "DQN", "ODT", "ODT_FINETUNED"]
queuing_idx = 11
prop_idx = 12

for scenario in scenarios:
    fig = go.Figure()
    has_data = False
    for agent in agents:
        path = f"{agent}_observations_{scenario}.csv"
        if not os.path.exists(path):
            continue
        data = pd.read_csv(path).values
        total_latency_ms = (data[:, queuing_idx] + data[:, prop_idx]) 
        total_latency_ms = total_latency_ms[total_latency_ms < 1000.0]
        if total_latency_ms.size == 0:
            continue
        fig.add_trace(go.Box(y=total_latency_ms, name=agent, boxmean=True))
        has_data = True
    if not has_data:
        continue
    fig.update_layout(
        title=f"Total Latency (Queuing + Propagation) - {scenario}",
        yaxis_title='Latency (ms)',
        xaxis_title='Agent',
        template='plotly_white'
    )
    fig.show()


In [14]:
import os
import pandas as pd
import numpy as np
import plotly.graph_objects as go

scenarios = ["no_scenario", "load_cycle_1", "load_cycle_2", "load_cycle_5", "medium_aircraft", "large_aircraft", "snr_congested"]
agents = ["BASELINE", "PPO", "DQN", "ODT", "ODT_FINETUNED"]
queuing_idx = 11
prop_idx = 12
latency_req_idx = 14

rows = []
for scenario in scenarios:
    for agent in agents:
        path = f"{agent}_observations_{scenario}.csv"
        if not os.path.exists(path):
            continue
        data = pd.read_csv(path).values
        total_latency_s = data[:, queuing_idx] + data[:, prop_idx]
        req_s = data[:, latency_req_idx]
        valid = req_s > 0
        if not valid.any():
            continue
        compliance = np.mean(total_latency_s[valid] <= req_s[valid])
        rows.append({
            'scenario': scenario,
            'agent': agent,
            'compliance_rate': compliance
        })

df = pd.DataFrame(rows)
df

for scenario in scenarios:
    subset = df[df['scenario'] == scenario]
    if subset.empty:
        continue
    fig = go.Figure()
    fig.add_trace(go.Bar(x=subset['agent'], y=subset['compliance_rate']))
    fig.update_layout(
        title=f"Latency Compliance Rate - {scenario}",
        xaxis_title='Agent',
        yaxis_title='Compliance Rate',
        yaxis=dict(range=[0, 1]),
        template='plotly_white'
    )
    fig.show()


In [15]:
import os
import pandas as pd
import numpy as np
import plotly.graph_objects as go

scenario = 'large_aircraft'
agent = 'BASELINE'
snr_idx = 4
load_idx = 5 
allocated_idx = 7
demand_idx = 9
deltaT = 5.0

path = f"{agent}_observations_{scenario}.csv"
if not os.path.exists(path):
    raise FileNotFoundError(path)

data = pd.read_csv(path).values
snr_db = data[:, snr_idx]
allocated_mbps = data[:, allocated_idx] * 8 / deltaT
demand_mbps = data[:, demand_idx] * 8 / deltaT
load = data[:, load_idx]
effective_bw_hz = (1 - load) * 250e6
shannon_capacity_mbps = np.minimum(effective_bw_hz * np.log2(1 + 10**(snr_db / 10)) / 1e6, 150) 

fig = go.Figure()
fig.add_trace(go.Scatter(y=allocated_mbps, mode='lines', name='Allocated (Mbps)'))
fig.add_trace(go.Scatter(y=shannon_capacity_mbps, mode='lines', name='Capacity (Mbps)'))
fig.add_trace(go.Scatter(y=demand_mbps, mode='lines', name='Demand (Mbps)'))
fig.add_trace(go.Scatter(y=effective_bw_hz, mode='lines', name='Effective BW', yaxis='y2'))

fig.update_layout(
    title=f"SNR vs Allocated/Capacity/Demand - {agent} ({scenario})",
    xaxis_title='Step',
    yaxis=dict(title='Mbps'),
    yaxis2=dict(title='SNR (dB)', overlaying='y', side='right'),
    template='plotly_white'
)
fig.show()
